In [ ]:
from PIL import Image
import sys

sys.path.insert(0, "../")
from src.model.gigachat_vl import GigaChatVL, GigaChatVLForInference, IGNORE_INDEX
from src.dataset.collator import VLMDataCollator
from src.utils.train_utils import save_artifacts
from transformers import Trainer, TrainingArguments
from torch.utils.data import Dataset
import torch

In [ ]:
LLM_PATH = "/media/alexey/HDDLargeData/models/VLM/GigaChat3.1-10B-A1.8B-bf16"
VISION_PATH = "/media/alexey/HDDLargeData/models/VLM/Qwen2.5-VL-7B-Instruct/"

sample1 = {
    "image": "/home/alexey/Downloads/tg.jpg",
    "question": "Извлеки текст из изображения",
    "answer": "Document Liberation Own your content",
}

sample2 = {
    "image": "/home/alexey/Downloads/tg2.jpg",
    "question": "Извлеки текст из изображения",
    "answer": "Привет, что делаешь? Вид, что всё хорошо.",
}

In [ ]:
class TwoSampleDataset(Dataset):
    def __init__(self, repeats=200):
        self.samples = [sample1, sample2] * repeats

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


model = GigaChatVL(
    llm_name=LLM_PATH,
    vision_name=VISION_PATH,
    use_4bit_llm=True,
    freeze_vision=True,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
)

collator = VLMDataCollator(model=model, max_length=256)
train_dataset = TwoSampleDataset(repeats=200)

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

args = TrainingArguments(
    output_dir="/tmp/gigachat_vl_two_image_test",
    max_steps=120,
    learning_rate=5e-4,
    weight_decay=0.0,
    warmup_steps=0,
    lr_scheduler_type="constant",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    logging_steps=10,
    save_strategy="no",
    bf16=use_bf16,
    fp16=use_fp16,
    remove_unused_columns=False,
    report_to="none",
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    data_collator=collator,
)

trainer.train()
save_artifacts(model, "/tmp/gigachat_vl_two_image_export")

In [ ]:
infer_model = GigaChatVLForInference(
    checkpoint_dir="/tmp/gigachat_vl_two_image_export",
    llm_name=LLM_PATH,
    vision_name=VISION_PATH,
    use_4bit_llm=True,
)

img1 = Image.open("/home/alexey/Downloads/tg.jpg").convert("RGB")
img2 = Image.open("/home/alexey/Downloads/tg2.jpg").convert("RGB")
blank = Image.new("RGB", img1.size, "white")

prompt = "Извлеки текст из изображения"

print("IMG1:")
print(infer_model.inference(prompt, image=img1, do_sample=False, max_new_tokens=48))
print()

print("IMG2:")
print(infer_model.inference(prompt, image=img2, do_sample=False, max_new_tokens=48))
print()

print("BLANK:")
print(infer_model.inference(prompt, image=blank, do_sample=False, max_new_tokens=48))
print()

print("NONE:")
print(infer_model.inference(prompt, image=None, do_sample=False, max_new_tokens=48))

In [ ]:
# Test raw GigaChat

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import sys

sys.path.insert(0, "../")

from src.model.gigachat_vl import _load_fast_tokenizer

MODEL_PATH = "/media/alexey/HDDLargeData/models/VLM/GigaChat3.1-10B-A1.8B-bf16"

tokenizer = _load_fast_tokenizer(MODEL_PATH)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)

In [ ]:
PROMPT = "Привет! Кратко представься и ответь, сколько будет 2+2."


prompt = (
    tokenizer.apply_chat_template(
        [{"role": "user", "content": PROMPT}],
        tokenize=False,
        add_generation_prompt=True,
    )
    if getattr(tokenizer, "chat_template", None)
    else f"User: {PROMPT}\nAssistant:"
)

print(prompt)

inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
inputs = {
    k: v.to(model.get_input_embeddings().weight.device) for k, v in inputs.items()
}

with torch.inference_mode():
    out = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=True,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

answer = tokenizer.decode(
    out[0, inputs["input_ids"].shape[1] :], skip_special_tokens=True
).strip()
print(answer)

In [ ]:
# Download datasets

In [ ]:
import sys

sys.path.insert(0, "../")

from src.dataset.sources.captioning.llava_pretrain_ru import download_llava_pretrain_ru

In [ ]:
download_llava_pretrain_ru(
    dataset_root="/media/alexey/HDDLargeData/datasets/llm/VL/Maya/"
)